# Explain Existing AMEX Model (Beyond SHAP, No Retraining)

This notebook is designed to explain already-trained artifacts from `amex_default_prediction.ipynb`.

It does **not train a new model**. It loads saved models and analyzes behavior using:

- Decile concentration and error diagnostics
- Global and segment permutation importance
- ALE (Accumulated Local Effects)
- ICE curves (local sensitivity)
- Pairwise interaction strength (H-stat approximation)
- Surrogate tree rules for stakeholder-friendly policy summaries

Supported trained artifacts:
- `v1`: `models/lightgbm_baseline.pkl`
- `v2`: `models/advanced_v2/branch_a_final.pkl`, `branch_b_final.pkl`, `meta_model.joblib`
- `v3`: `models/advanced_v3/lightgbm_v3.pkl` + `statement_meta_train.parquet`

Note: `v4` is not reloadable from current pipeline because only submission CSV is saved (no persisted model object).


## 1) Setup


In [ ]:
from pathlib import Path
import json
import itertools
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.tree import DecisionTreeRegressor, export_text

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 200)


## 2) Config


In [ ]:
# Set to one of: 'v1', 'v2', 'v3'
MODEL_VERSION = 'v2'

# Data root used by your main training notebook.
CANDIDATE_ROOTS = [
    Path('/content/drive/MyDrive/amex_data_parquet'),
    Path('../data/raw'),
    Path('data/raw'),
]
ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])

RAW_LAYER_DIR = ROOT
FEATURE_LAYER_DIR = ROOT / 'features'
MODEL_LAYER_DIR = ROOT / 'models'

TRAIN_CHUNK_DIR = FEATURE_LAYER_DIR / 'train_chunks'
FEATURE_META_PATH = FEATURE_LAYER_DIR / 'feature_meta.json'
MODEL_META_PATH = MODEL_LAYER_DIR / 'model_meta.json'

# Runtime controls
RANDOM_STATE = 42
EVAL_FOLDS = {0}             # consistent hash-based slice for diagnostics
MAX_EVAL_CUSTOMERS = 120_000 # sampled from selected folds if needed

# Explanation controls
PERM_TOP_FEATURES = 25
ALE_TOP_FEATURES = 6
ICE_TOP_FEATURES = 3
INTERACTION_TOP_FEATURE_POOL = 8

print('ROOT:', ROOT)
print('MODEL_VERSION:', MODEL_VERSION)


## 3) Paths and Utility Functions


In [ ]:
LABELS_PATH = RAW_LAYER_DIR / 'train_labels.csv'
LGB_MODEL_PATH = MODEL_LAYER_DIR / 'lightgbm_baseline.pkl'

ADV_V2_DIR = MODEL_LAYER_DIR / 'advanced_v2'
V2_A_PATH = ADV_V2_DIR / 'branch_a_final.pkl'
V2_B_PATH = ADV_V2_DIR / 'branch_b_final.pkl'
V2_META_PATH = ADV_V2_DIR / 'meta_model.joblib'
V2_CB_PATH = ADV_V2_DIR / 'catboost_meta.cbm'

ADV_V3_DIR = MODEL_LAYER_DIR / 'advanced_v3'
V3_MODEL_PATH = ADV_V3_DIR / 'lightgbm_v3.pkl'
STMT_TRAIN_META_PATH = ADV_V3_DIR / 'statement_meta_train.parquet'


def list_chunk_paths(out_dir: Path, prefix: str) -> list[Path]:
    return sorted(out_dir.glob(f'{prefix}_chunk_*.parquet'))


def rank01(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method='average', pct=True).values


def add_fold_id(customer_series: pd.Series, n_folds: int = 5) -> pd.Series:
    h = pd.util.hash_pandas_object(customer_series.astype('string'), index=False).astype('uint64')
    return (h % np.uint64(n_folds)).astype('int64')


def get_branch_cols(feature_cols_all: list[str]) -> tuple[list[str], list[str]]:
    cols_a = feature_cols_all[:]
    keep_suffix = ('_last', '_mean', '_std', '_missing_rate', '_last_cat', '_nunique_cat')
    keep_exact = {'recency_days', 'history_days', 'n_statements'}
    cols_b = [c for c in feature_cols_all if c.endswith(keep_suffix) or c in keep_exact]
    cols_b = sorted(list(set(cols_b)))
    if not cols_b:
        cols_b = cols_a[:]
    return cols_a, cols_b


def _prepare_lgb_frame(df: pd.DataFrame, cat_columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cat_columns:
        if c in out.columns:
            out[c] = out[c].astype('category')
    return out


train_chunk_paths = list_chunk_paths(TRAIN_CHUNK_DIR, 'train')

if not LABELS_PATH.exists():
    raise FileNotFoundError(f'Missing labels: {LABELS_PATH}')
if not train_chunk_paths:
    raise FileNotFoundError(f'No train feature chunks found under: {TRAIN_CHUNK_DIR}')
if not FEATURE_META_PATH.exists():
    raise FileNotFoundError(f'Missing feature metadata: {FEATURE_META_PATH}')

labels_map = pd.read_csv(LABELS_PATH)[['customer_ID', 'target']]
labels_map['customer_ID'] = labels_map['customer_ID'].astype('string')
labels_map['target'] = labels_map['target'].astype(int)

feature_meta = json.loads(FEATURE_META_PATH.read_text())
base_feature_cols = list(feature_meta['feature_columns'])
base_cat_cols = list(feature_meta.get('categorical_columns', []))

print('train chunks:', len(train_chunk_paths))
print('labels rows:', len(labels_map))
print('base feature count:', len(base_feature_cols))


## 4) Load Trained Model Artifacts


In [ ]:
model_bundle = {'version': MODEL_VERSION}

if MODEL_VERSION == 'v1':
    if not LGB_MODEL_PATH.exists():
        raise FileNotFoundError(f'Missing v1 model: {LGB_MODEL_PATH}')

    gbm = joblib.load(LGB_MODEL_PATH)

    if MODEL_META_PATH.exists():
        mm = json.loads(MODEL_META_PATH.read_text())
        model_feature_cols = list(mm['feature_columns'])
        model_cat_cols = list(mm.get('categorical_columns', []))
    else:
        model_feature_cols = base_feature_cols
        model_cat_cols = base_cat_cols

    model_bundle.update({
        'estimator': gbm,
        'feature_cols': model_feature_cols,
        'cat_cols': model_cat_cols,
        'score_kind': 'prob',
    })

elif MODEL_VERSION == 'v2':
    missing = [p for p in [V2_A_PATH, V2_B_PATH, V2_META_PATH] if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Missing v2 artifacts: {[str(x) for x in missing]}')

    m_a = joblib.load(V2_A_PATH)
    m_b = joblib.load(V2_B_PATH)
    meta_model = joblib.load(V2_META_PATH)

    cb_meta_model = None
    try:
        from catboost import CatBoostClassifier
        if V2_CB_PATH.exists():
            cb_meta_model = CatBoostClassifier(verbose=False)
            cb_meta_model.load_model(str(V2_CB_PATH))
    except Exception:
        cb_meta_model = None

    if MODEL_META_PATH.exists():
        mm = json.loads(MODEL_META_PATH.read_text())
        feature_cols_all = list(mm['feature_columns'])
        cat_cols_all = list(mm.get('categorical_columns', []))
    else:
        feature_cols_all = base_feature_cols
        cat_cols_all = base_cat_cols

    branch_a_cols, branch_b_cols = get_branch_cols(feature_cols_all)
    model_feature_cols = sorted(list(set(branch_a_cols + branch_b_cols)))

    model_bundle.update({
        'm_a': m_a,
        'm_b': m_b,
        'meta_model': meta_model,
        'cb_meta_model': cb_meta_model,
        'branch_a_cols': branch_a_cols,
        'branch_b_cols': branch_b_cols,
        'feature_cols': model_feature_cols,
        'cat_cols': cat_cols_all,
        'score_kind': 'meta_prob',
    })

elif MODEL_VERSION == 'v3':
    if not V3_MODEL_PATH.exists():
        raise FileNotFoundError(f'Missing v3 model: {V3_MODEL_PATH}')
    if not STMT_TRAIN_META_PATH.exists():
        raise FileNotFoundError(
            f'Missing v3 statement meta features: {STMT_TRAIN_META_PATH}\n'
            'Re-run v3 stage in amex_default_prediction.ipynb to generate cached statement meta features.'
        )

    v3_booster = joblib.load(V3_MODEL_PATH)
    stmt_meta_train = pd.read_parquet(STMT_TRAIN_META_PATH)
    stmt_meta_train['customer_ID'] = stmt_meta_train['customer_ID'].astype('string')
    stmt_meta_cols = [c for c in stmt_meta_train.columns if c != 'customer_ID']

    v3_feature_cols = sorted(list(set(base_feature_cols + stmt_meta_cols)))

    model_bundle.update({
        'estimator': v3_booster,
        'stmt_meta_train': stmt_meta_train,
        'feature_cols': v3_feature_cols,
        'cat_cols': [c for c in base_cat_cols if c in v3_feature_cols],
        'stmt_meta_cols': stmt_meta_cols,
        'score_kind': 'prob',
    })

else:
    raise ValueError("MODEL_VERSION must be one of: 'v1', 'v2', 'v3'")

print('Loaded model bundle for', MODEL_VERSION)
print('model feature count:', len(model_bundle['feature_cols']))


## 5) Build Evaluation Frame from Existing Feature Chunks (No Retraining)


In [ ]:
n_labels = labels_map['customer_ID'].nunique()
sample_frac = min(1.0, MAX_EVAL_CUSTOMERS / max(n_labels, 1))

# Stable hash sampling by customer.
label_hash = pd.util.hash_pandas_object(labels_map['customer_ID'], index=False).astype('uint64')
sample_mask = (label_hash % np.uint64(10_000)) < np.uint64(int(sample_frac * 10_000))
sampled_ids = set(labels_map.loc[sample_mask, 'customer_ID'].astype('string').tolist())

eval_rows = []

need_cols = set(model_bundle['feature_cols']) | {'customer_ID', 'last_statement_date'}

for cp in train_chunk_paths:
    cdf = pd.read_parquet(cp)
    if cdf.empty:
        continue

    cdf['customer_ID'] = cdf['customer_ID'].astype('string')

    # Attach labels if not present in chunk.
    if 'target' not in cdf.columns:
        cdf = cdf.merge(labels_map, on='customer_ID', how='left')

    cdf = cdf[cdf['target'].notna()].copy()
    if cdf.empty:
        continue

    # Keep only configured folds and sampled IDs.
    cdf['fold_id'] = add_fold_id(cdf['customer_ID'])
    cdf = cdf[cdf['fold_id'].isin(EVAL_FOLDS)]
    cdf = cdf[cdf['customer_ID'].isin(sampled_ids)]

    if cdf.empty:
        continue

    for col in need_cols:
        if col not in cdf.columns:
            cdf[col] = np.nan

    keep = list(need_cols | {'target', 'fold_id'})

    if MODEL_VERSION == 'v3':
        stmt_meta_train = model_bundle['stmt_meta_train']
        cdf = cdf[keep].merge(stmt_meta_train, on='customer_ID', how='left')
        # Ensure all v3 model cols exist post-merge.
        for col in model_bundle['feature_cols']:
            if col not in cdf.columns:
                cdf[col] = np.nan

    eval_rows.append(cdf)

if not eval_rows:
    raise RuntimeError('No evaluation rows found. Check ROOT path, feature chunks, and fold/sample settings.')

eval_df = pd.concat(eval_rows, axis=0, ignore_index=True)
eval_df = eval_df.drop_duplicates(subset=['customer_ID'], keep='last').reset_index(drop=True)

print('evaluation customers:', len(eval_df))
print('evaluation default rate:', f"{eval_df['target'].mean():.3%}")


## 6) Score Evaluation Customers with Loaded Model


In [ ]:
feature_cols = model_bundle['feature_cols']
cat_cols = model_bundle.get('cat_cols', [])

X_eval = eval_df[feature_cols].copy()
y_eval = eval_df['target'].astype(int).copy()

# Keep default values for missing columns when we later perturb subsets.
feature_defaults = {}
for c in feature_cols:
    s = X_eval[c]
    if pd.api.types.is_numeric_dtype(s):
        med = s.median(skipna=True)
        feature_defaults[c] = float(med) if pd.notna(med) else 0.0
    else:
        mode = s.mode(dropna=True)
        feature_defaults[c] = str(mode.iloc[0]) if len(mode) > 0 else '__MISSING__'


def _complete_features(X_partial: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=X_partial.index)
    for c in feature_cols:
        if c in X_partial.columns:
            out[c] = X_partial[c]
        else:
            out[c] = feature_defaults[c]
    return out


def predict_model_proba(X_partial: pd.DataFrame) -> np.ndarray:
    X_full = _complete_features(X_partial)

    if MODEL_VERSION in ('v1', 'v3'):
        est = model_bundle['estimator']
        Xp = _prepare_lgb_frame(X_full, [c for c in cat_cols if c in X_full.columns])
        pred = est.predict(Xp)
        return np.asarray(pred).reshape(-1)

    # v2 ensemble scoring
    m_a = model_bundle['m_a']
    m_b = model_bundle['m_b']
    meta_model = model_bundle['meta_model']
    cb_meta_model = model_bundle.get('cb_meta_model', None)
    branch_a_cols = model_bundle['branch_a_cols']
    branch_b_cols = model_bundle['branch_b_cols']

    Xa = _prepare_lgb_frame(X_full[branch_a_cols], [c for c in cat_cols if c in branch_a_cols])
    Xb = _prepare_lgb_frame(X_full[branch_b_cols], [c for c in cat_cols if c in branch_b_cols])

    pa = np.asarray(m_a.predict(Xa)).reshape(-1)
    pb = np.asarray(m_b.predict(Xb)).reshape(-1)

    meta_chunk = pd.DataFrame({
        'pred_a': pa,
        'pred_b': pb,
        'pred_rank_mean': 0.5 * (rank01(pa) + rank01(pb)),
    })

    pmeta = meta_model.predict_proba(meta_chunk)[:, 1]

    # For explainability surfaces (ALE/ICE), we use probability-like score.
    # If CatBoost meta exists, blend with it in probability space.
    if cb_meta_model is not None:
        pcb = cb_meta_model.predict_proba(meta_chunk)[:, 1]
        return 0.5 * pmeta + 0.5 * pcb

    return pmeta


pred_eval = predict_model_proba(X_eval)

auc = roc_auc_score(y_eval, pred_eval)
ap = average_precision_score(y_eval, pred_eval)
brier = brier_score_loss(y_eval, pred_eval)

print(f'Eval ROC-AUC: {auc:.4f}')
print(f'Eval PR-AUC : {ap:.4f}')
print(f'Eval Brier  : {brier:.4f}')


## 7) Decile Diagnostics


In [ ]:
scored = pd.DataFrame({
    'customer_ID': eval_df['customer_ID'].astype('string'),
    'target': y_eval.values,
    'pred': pred_eval,
})

scored['risk_decile'] = pd.qcut(scored['pred'].rank(method='first'), 10, labels=False) + 1

decile = scored.groupby('risk_decile').agg(
    customers=('target', 'size'),
    avg_pred=('pred', 'mean'),
    default_rate=('target', 'mean')
).reset_index()

base = scored['target'].mean()
decile['lift'] = decile['default_rate'] / base
decile = decile.sort_values('risk_decile', ascending=False)

decile


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=decile, x='risk_decile', y='default_rate', ax=axes[0], color='#C44E52')
axes[0].set_title('Observed Default Rate by Risk Decile')
axes[0].set_xlabel('Risk decile (10 = highest)')

sns.barplot(data=decile, x='risk_decile', y='lift', ax=axes[1], color='#4C72B0')
axes[1].set_title('Lift by Risk Decile')
axes[1].set_xlabel('Risk decile (10 = highest)')

plt.tight_layout()
plt.show()


## 8) Native Feature Priority Seed (from trained model artifacts)


In [ ]:
seed_features = []

if MODEL_VERSION in ('v1', 'v3'):
    est = model_bundle['estimator']
    try:
        names = list(est.feature_name())
    except Exception:
        names = feature_cols

    try:
        imp = np.asarray(est.feature_importance(importance_type='gain'))
    except Exception:
        imp = np.ones(len(names), dtype=float)

    native_imp = pd.DataFrame({'feature': names, 'importance': imp}).sort_values('importance', ascending=False)
    seed_features = native_imp['feature'].tolist()

elif MODEL_VERSION == 'v2':
    m_a = model_bundle['m_a']
    m_b = model_bundle['m_b']

    try:
        na = list(m_a.feature_name())
        ia = np.asarray(m_a.feature_importance(importance_type='gain'))
        da = pd.DataFrame({'feature': na, 'importance_a': ia})
    except Exception:
        da = pd.DataFrame({'feature': model_bundle['branch_a_cols'], 'importance_a': 1.0})

    try:
        nb = list(m_b.feature_name())
        ib = np.asarray(m_b.feature_importance(importance_type='gain'))
        db = pd.DataFrame({'feature': nb, 'importance_b': ib})
    except Exception:
        db = pd.DataFrame({'feature': model_bundle['branch_b_cols'], 'importance_b': 1.0})

    merged = da.merge(db, on='feature', how='outer').fillna(0.0)
    merged['importance'] = merged['importance_a'] + merged['importance_b']
    native_imp = merged.sort_values('importance', ascending=False)
    seed_features = native_imp['feature'].tolist()

# Keep numeric features only for robust perturbation plots.
numeric_cols_eval = [c for c in X_eval.columns if pd.api.types.is_numeric_dtype(X_eval[c])]
seed_features = [c for c in seed_features if c in numeric_cols_eval]

candidate_features = seed_features[:max(40, PERM_TOP_FEATURES)]
print('candidate feature count:', len(candidate_features))
pd.DataFrame({'candidate_feature': candidate_features}).head(20)


## 9) Global Permutation Importance


In [ ]:
class LoadedModelEstimator:
    def __init__(self, predict_fn):
        self._predict_fn = predict_fn
        self.classes_ = np.array([0, 1])
    def predict_proba(self, X):
        p = self._predict_fn(pd.DataFrame(X, columns=getattr(X, 'columns', None), index=getattr(X, 'index', None)))
        p = np.asarray(p).reshape(-1)
        return np.column_stack([1 - p, p])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


estimator = LoadedModelEstimator(predict_model_proba)

X_perm = X_eval[candidate_features].copy()

perm = permutation_importance(
    estimator,
    X_perm,
    y_eval,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring='roc_auc',
    n_jobs=1,
)

perm_df = pd.DataFrame({
    'feature': X_perm.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

top_perm = perm_df.head(PERM_TOP_FEATURES)
top_perm


In [ ]:
plt.figure(figsize=(8, max(4, 0.3 * len(top_perm))))
sns.barplot(data=top_perm, y='feature', x='importance_mean', color='#2A9D8F')
plt.title('Permutation Importance (AUC drop)')
plt.xlabel('mean importance')
plt.ylabel('feature')
plt.tight_layout()
plt.show()


## 10) Segment-Specific Importance (High-Risk vs Low-Risk)


In [ ]:
scored_local = scored.copy()
hi_idx = scored_local['risk_decile'] >= 9
lo_idx = scored_local['risk_decile'] <= 2

def segment_perm(mask):
    Xs = X_perm.loc[mask].copy()
    ys = y_eval.loc[mask]
    if len(Xs) < 1200 or ys.nunique() < 2:
        return None
    out = permutation_importance(
        estimator,
        Xs,
        ys,
        n_repeats=4,
        random_state=RANDOM_STATE,
        scoring='roc_auc',
        n_jobs=1,
    )
    return pd.DataFrame({'feature': Xs.columns, 'importance': out.importances_mean})

hi_imp = segment_perm(hi_idx)
lo_imp = segment_perm(lo_idx)

if hi_imp is None or lo_imp is None:
    print('Not enough data for stable segment comparison.')
else:
    seg_cmp = hi_imp.merge(lo_imp, on='feature', suffixes=('_highrisk', '_lowrisk'))
    seg_cmp['delta'] = seg_cmp['importance_highrisk'] - seg_cmp['importance_lowrisk']
    seg_cmp = seg_cmp.sort_values('delta', ascending=False)
    seg_cmp.head(20)


## 11) ALE Curves


In [ ]:
def ale_1d(predict_fn, X, feature, bins=20):
    x = X[feature].values
    q = np.quantile(x, np.linspace(0, 1, bins + 1))
    edges = np.unique(q)
    if len(edges) < 4:
        return None

    effects, counts, centers = [], [], []

    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        if i < len(edges) - 2:
            idx = np.where((x >= lo) & (x < hi))[0]
        else:
            idx = np.where((x >= lo) & (x <= hi))[0]

        if len(idx) == 0:
            effects.append(0.0)
            counts.append(0)
            centers.append((lo + hi) / 2)
            continue

        Xl = X.iloc[idx].copy()
        Xh = X.iloc[idx].copy()
        Xl[feature] = lo
        Xh[feature] = hi

        diff = predict_fn(Xh) - predict_fn(Xl)
        effects.append(float(np.mean(diff)))
        counts.append(len(idx))
        centers.append((lo + hi) / 2)

    ale = np.cumsum(np.array(effects, dtype=float))
    w = np.array(counts, dtype=float)
    if w.sum() > 0:
        ale = ale - np.average(ale, weights=w)

    return pd.DataFrame({'feature': feature, 'value': centers, 'ale': ale, 'count': counts})


top_for_ale = top_perm['feature'].head(ALE_TOP_FEATURES).tolist()
X_ale = X_eval[top_for_ale].copy()
ale_frames = []

for f in top_for_ale:
    # ALE evaluation uses full model via predict_model_proba + partial columns.
    out = ale_1d(predict_model_proba, X_ale[[f]].join(X_eval.drop(columns=[f], errors='ignore')), f, bins=20)
    if out is not None:
        ale_frames.append(out)

if ale_frames:
    ale_df = pd.concat(ale_frames, ignore_index=True)
    n = len(top_for_ale)
    fig, axes = plt.subplots(n, 1, figsize=(7, max(3, 2.8*n)))
    if n == 1:
        axes = [axes]
    for ax, f in zip(axes, top_for_ale):
        tmp = ale_df[ale_df['feature'] == f]
        sns.lineplot(data=tmp, x='value', y='ale', marker='o', ax=ax, color='#264653')
        ax.axhline(0, ls='--', c='gray', lw=1)
        ax.set_title(f'ALE: {f}')
        ax.set_xlabel(f)
        ax.set_ylabel('ALE effect')
    plt.tight_layout()
    plt.show()
else:
    print('ALE unavailable for selected features.')


## 12) ICE Curves (Local Sensitivity)


In [ ]:
top_for_ice = top_perm['feature'].head(ICE_TOP_FEATURES).tolist()

n_profiles = min(40, len(X_eval))
profile_idx = X_eval.sample(n=n_profiles, random_state=RANDOM_STATE).index

for f in top_for_ice:
    grid = np.quantile(X_eval[f], np.linspace(0.05, 0.95, 20))
    grid = np.unique(grid)

    rows = []
    for idx in profile_idx:
        base_row = X_eval.loc[[idx]].copy()
        for g in grid:
            row = base_row.copy()
            row[f] = g
            p = float(predict_model_proba(row)[0])
            rows.append((idx, g, p))

    ice_df = pd.DataFrame(rows, columns=['profile', 'value', 'pred'])

    plt.figure(figsize=(7, 4))
    for pid, grp in ice_df.groupby('profile'):
        plt.plot(grp['value'], grp['pred'], color='gray', alpha=0.25, lw=1)

    pdp = ice_df.groupby('value', as_index=False)['pred'].mean()
    plt.plot(pdp['value'], pdp['pred'], color='#D62828', lw=2.5, label='PDP mean')
    plt.title(f'ICE + PDP (manual): {f}')
    plt.xlabel(f)
    plt.ylabel('predicted default risk')
    plt.legend()
    plt.tight_layout()
    plt.show()


## 13) Pairwise Interaction Strength (Approx. H-Statistic)


In [ ]:
def interaction_h_approx(predict_fn, X_base, f1, f2, grid_n=12):
    g1 = np.quantile(X_base[f1], np.linspace(0.05, 0.95, grid_n))
    g2 = np.quantile(X_base[f2], np.linspace(0.05, 0.95, grid_n))
    g1 = np.unique(g1)
    g2 = np.unique(g2)

    if len(g1) < 4 or len(g2) < 4:
        return np.nan

    z = np.zeros((len(g1), len(g2)), dtype=float)
    for i, v1 in enumerate(g1):
        for j, v2 in enumerate(g2):
            tmp = X_base.copy()
            tmp[f1] = v1
            tmp[f2] = v2
            z[i, j] = float(np.mean(predict_fn(tmp)))

    f1_curve = z.mean(axis=1)
    f2_curve = z.mean(axis=0)
    additive = f1_curve[:, None] + f2_curve[None, :] - z.mean()
    inter = z - additive

    denom = np.var(z)
    if denom <= 1e-12:
        return 0.0

    h2 = np.var(inter) / denom
    return float(np.sqrt(max(0.0, h2)))


pair_pool = top_perm['feature'].head(INTERACTION_TOP_FEATURE_POOL).tolist()
pair_list = list(itertools.combinations(pair_pool, 2))

X_inter = X_eval.sample(n=min(2000, len(X_eval)), random_state=RANDOM_STATE).copy()

inter_rows = []
for f1, f2 in pair_list:
    try:
        h = interaction_h_approx(predict_model_proba, X_inter, f1, f2, grid_n=10)
        inter_rows.append((f1, f2, h))
    except Exception:
        continue

inter_df = pd.DataFrame(inter_rows, columns=['feature_1', 'feature_2', 'h_stat'])
inter_df = inter_df.dropna().sort_values('h_stat', ascending=False)

top_inter = inter_df.head(15).copy()
top_inter['pair'] = top_inter['feature_1'] + ' x ' + top_inter['feature_2']
top_inter


In [ ]:
if len(top_inter) > 0:
    plt.figure(figsize=(8, max(4, 0.35 * len(top_inter))))
    sns.barplot(data=top_inter, y='pair', x='h_stat', color='#8AB17D')
    plt.title('Top Pairwise Interactions (Approx. H-stat)')
    plt.xlabel('interaction strength')
    plt.ylabel('feature pair')
    plt.tight_layout()
    plt.show()
else:
    print('No interaction estimates available.')


## 14) Surrogate Tree Rules


In [ ]:
surrogate_features = top_perm['feature'].head(20).tolist()
X_sur = X_eval[surrogate_features].copy()

surrogate = DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=max(150, int(0.01 * len(X_sur))),
    random_state=RANDOM_STATE,
)
surrogate.fit(X_sur, pred_eval)

r2 = surrogate.score(X_sur, pred_eval)
print(f'Surrogate R^2 vs model score: {r2:.4f}')

rules = export_text(surrogate, feature_names=surrogate_features, max_depth=3)
print(rules)


## 15) Interpretation Checklist


Use this summary structure after running the notebook:

1. Top global drivers (Permutation):
2. Drivers that change in high-risk segment:
3. Key ALE thresholds / nonlinear effects:
4. Strongest feature interactions:
5. Surrogate rules for stakeholder communication:
6. Decile-level model risk observations (over/under concentration):

Important caveat:
- If you explain scores on training-derived customers (without strict OOF cache), treat metrics as **diagnostic**, not final generalization estimates.
